In [ ]:
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report
import tqdm
from ST_GCN_v2 import LateFusion_STGCN  # Ensure this matches your actual model class

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import glob
import os
import joblib

In [ ]:
# Configuration
F_DATA_PATH        = 'split_data/front_view'  # thư mục chứa các file .npz        
L_DATA_PATH        = 'split_data/left_view'   # thư mục chứa các file .npz
R_DATA_PATH        = 'split_data/right_view'  # thư mục chứa các file .npz
LABEL_MAP_PATH   = 'Logs/label_map.json'  # đường dẫn đến file label_map.json
BATCH_SIZE       = 128
EPOCHS           = 100
LEARNING_RATE    = 1e-3
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Load Files and Labels
with open(LABEL_MAP_PATH, 'r', encoding='utf-8') as f:
    label_map = json.load(f)

train_files = glob.glob(os.path.join(F_DATA_PATH, 'train', '**', '*.npz'), recursive=True)
val_files   = glob.glob(os.path.join(F_DATA_PATH, 'val', '**', '*.npz'), recursive=True)


class VSL_GCN_Dataset(Dataset):
    def __init__(self, file_list):
        self.files = file_list
        # Define the parent map for bones (MediaPipe Hand)
        self.parents = [0, 0, 1, 2, 3, 0, 5, 6, 7, 0, 9, 10, 11, 0, 13, 14, 15, 0, 17, 18, 19]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f_path = self.files[idx]
        
        # 1. Load raw sequence (assuming 60 frames, 64 features)
        # We need to extract the 42 keypoints (21 left, 21 right)
        # Note: Adjust the indexing [:42*3] based on how your .npz is saved!
        raw_data = np.load(f_path)["sequence"].astype(np.float32) # (60, 64)
        
        # Reshape to (Time, Nodes, Channels) -> (60, 42, 3)
        # This assumes your first 126 features are (x,y,z * 42)
        # If your data is 2D, adjust channels to 2.
        # Here we slice and reshape.
        joints = raw_data[:, :126].reshape(60, 42, 3) 
        
        # Transpose to ST-GCN format: (Channels, Time, Nodes) -> (3, 60, 42)
        x_joint = torch.from_numpy(joints).permute(2, 0, 1)

        if self.is_train:
            # 0.005 is roughly 0.5% coordinate shift - enough to 
            # prevent memorization without ruining the sign shape.
            std = 0.005 
            noise = torch.randn_like(x_joint) * std
            x_joint = x_joint + noise

        # 2. Calculate Bone Stream (Relative distance)
        x_bone = torch.zeros_like(x_joint)
        for i in range(21):
            # Left Hand
            x_bone[:, :, i] = x_joint[:, :, i] - x_joint[:, :, self.parents[i]]
            # Right Hand
            x_bone[:, :, i+21] = x_joint[:, :, i+21] - x_joint[:, :, self.parents[i]+21]

        # 3. Calculate Motion Stream (Temporal difference)
        x_motion = torch.zeros_like(x_joint)
        x_motion[:, 1:, :] = x_joint[:, 1:, :] - x_joint[:, :-1, :]

        label = int(np.load(f_path)["label"])
        
        return x_joint, x_bone, x_motion, label

# 3. Simplified DataLoaders
# Batch size can be larger for GCN (e.g., 64 or 128)

train_set = VSL_GCN_Dataset(train_files, is_train=True)
val_set = VSL_GCN_Dataset(val_files, is_train=False)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_set, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)         

In [ ]:
def evaluate_stgcn_f1(model_path, val_loader, device, num_classes=400):
    # 1. Initialize the ST-GCN Model
    # Use the same class name from your training reference
    model = LateFusion_STGCN(num_classes=num_classes)
    
    # Load weights - Using weights_only=True is safer in recent PyTorch versions
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    print(f"🔍 Testing ST-GCN Branch: {model_path}")
    
    with torch.no_grad():
        # Unpack only what we need (skipping phonetic views batch[0:3])
        for batch in tqdm.tqdm(val_loader, desc="ST-GCN Inference"):
            # batch[3]=joints, batch[4]=bones, batch[5]=motion, batch[6]=labels
            x_j = batch[3].to(device)
            x_b = batch[4].to(device)
            x_m = batch[5].to(device)
            labels = batch[6].to(device)

            with torch.cuda.amp.autocast():
                # Forward pass for the kinematic branch only
                outputs = model(x_j, x_b, x_m)
                preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 2. Calculate Metrics
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\n--- 📈 ST-GCN Evaluation Results ---")
    print(f"Macro F1-Score: {macro_f1:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds))
    
    return macro_f1

evaluate_stgcn_f1('best_stgcn_model.pth', val_loader, DEVICE)